# 挑战 02 — 数据质量：审计 → 修复 → 验证

**业务背景**：挑战 02 把挑战 01 的「有序数据」升级为「可信数据」。给定一套*结构化但充满噪音*的杂货零售数据库（5 张表），本笔记本系统化地完成：

1. **审计** — 穷举缺失、重复、外键孤儿、偏移类别、数值越界与金额不一致；
2. **修复** — 优先*精确还原*（从业务不变量反推），仅在无信号时做原则性补值；
3. **验证** — 证明清洗没有引入偏置、没有扭曲关键分布。

每一步都记录在变更日志中，可被他人复核。


## 0. Setup —— 环境与数据


In [3]:
import sys
from pathlib import Path
# locate project root robustly (works from notebooks/ or project root)
for _cand in [Path.cwd(), Path.cwd().parent]:
    if (_cand / 'src' / 'helper_utils.py').exists():
        BASE = _cand; break
sys.path.insert(0, str(BASE / 'src'))
FIG = BASE / 'reports' / 'figures'

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import helper_utils as hu

RAW = BASE / 'data' / 'raw'
CLEAN = BASE / 'data' / 'clean'
dfs = hu.load_raw(RAW)
for t, df in dfs.items():
    print(f'{t:<11} {df.shape[0]:>5} rows x {df.shape[1]} cols')


products       20 rows x 5 cols
vendors         5 rows x 2 cols
stores          5 rows x 3 cols
categories      5 rows x 2 cols
sales        2859 rows x 7 cols


## 1. Audit —— 错误审计

对 5 张表做完整性、重复、漂流、越界的量化盘点。`audit()` 只*检测计数*、绝不改动数据。


In [5]:
findings = hu.audit(dfs)
fdf = pd.DataFrame([(k, v) for k, v in findings.items()], columns=['finding', 'value'])
print(fdf.to_string(index=False))


                   finding               value
          missing_txn_date                  29
               missing_qty                  57
        missing_unit_price                  56
       missing_line_amount                   0
                dup_txn_id                  50
              dup_full_row                  47
             store_orphans                  14
           product_orphans                  29
          category_orphans                   0
       qty_outliers_ge1000                  29
           qty_nonpositive                   0
         price_nonpositive                   0
           amount_mismatch                  43
          unparseable_date                   0
                  date_min 2024-01-01 00:00:00
                  date_max 2024-02-29 00:00:00
           products_dup_id                   0
            vendors_dup_id                   0
             stores_dup_id                   0
         categories_dup_id                   0
category_name

### 关键发现（错误分类）

| 类别 | 计数 | 位置 |
|---|---|---|
| 缺失 `txn_date` | 29 | sales |
| 缺失 `qty` | 57 | sales |
| 缺失 `unit_price` | 56 | sales |
| 重复 `txn_id` | 50 组 / 100 行 | sales |
| 外键孤儿 `store_id=9999` | 14 | sales |
| 外键孤儿 `product_id=9999` | 29 | sales |
| 数量离群（1000–4000） | 29 | sales |
| 金额不一致（≠qty×price） | 43 | sales |
| 类别命名漂移 | 2 | categories |


### 缺失画像（missingness）


In [8]:
miss = pd.Series({
    'txn_date': findings['missing_txn_date'],
    'qty': findings['missing_qty'],
    'unit_price': findings['missing_unit_price'],
    'line_amount': findings['missing_line_amount']})
fig, ax = plt.subplots(figsize=(7, 3))
b = ax.barh(miss.index[::-1], miss.values[::-1], color=['#f0a32c', '#2c7fb8', '#2c7fb8', '#1b4f72'])
ax.bar_label(b, padding=2); ax.set_title('Sales — missing values per column (raw)')
ax.set_xlim(0, 60); plt.tight_layout()
plt.savefig(FIG / 'fig1_missing_values.png', dpi=110); plt.close()
print('saved', FIG / 'fig1_missing_values.png')


saved C:\Users\21597\Desktop\spider_redbook\challenge_02_errors_integrity\reports\figures\fig1_missing_values.png


![缺失画像](../reports/figures/fig1_missing_values.png)

三列都有缺失，但**没有任何一行的 `line_amount` 缺失**——这是后续能精确还原 qty/price 的关键。


### 数量离群（spike）


In [11]:
raw_q = pd.to_numeric(dfs['sales']['qty'], errors='coerce')
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.4))
vcr = raw_q.dropna().value_counts().sort_index()
a1.bar(vcr.index.astype(str), vcr.values, color='#2c7fb8')
a1.set_title('Raw qty (spikes 1000–4000)'); a1.set_xlabel('qty')
a2.bar(['1','2','3','4'], [raw_q.isin([1,2,3,4]).sum()] + list(raw_q[raw_q.isin([2,3,4])].value_counts().sort_index().values), color='#1b4f72')
a2.set_title('Clean domain (1–4 only)'); a2.set_xlabel('qty')
plt.tight_layout(); plt.savefig(FIG / 'fig2_qty_spikes.png', dpi=110); plt.close()
print('qty outlier rows:', int((raw_q >= 1000).sum()))


qty outlier rows: 29


![数量离群](../reports/figures/fig2_qty_spikes.png)


### 金额一致性（ratio = line_amount / (qty×price)）


In [14]:
s = dfs['sales'].copy()
s['q'] = pd.to_numeric(s['qty'], errors='coerce')
s['p'] = pd.to_numeric(s['unit_price'], errors='coerce')
s['a'] = pd.to_numeric(s['line_amount'], errors='coerce')
s['r'] = s['a'] / (s['q'] * s['p'])
r = s['r'].dropna(); r = r[(r > 0) & np.isfinite(np.log10(r))]
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.hist(np.log10(r), bins=60, color='#2c7fb8', edgecolor='white')
ax.axvline(0, color='#1b4f72', ls='--', lw=1.2, label='consistent (ratio=1)')
ax.axvline(2, color='#c0392b', ls=':', lw=1.2, label='x100 corruption')
ax.axvline(-3, color='#f0a32c', ls=':', lw=1.2, label='qty x1000 (amount ok)')
ax.set_xlabel('log10(line_amount / (qty*unit_price))'); ax.set_ylabel('rows')
ax.legend(fontsize=8); plt.tight_layout()
plt.savefig(FIG / 'fig3_amount_ratio.png', dpi=110); plt.close()
print('mismatch rows:', int((s['a'] - s['q'] * s['p']).abs().gt(1e-2).sum()))


mismatch rows: 43


![金额一致性](../reports/figures/fig3_amount_ratio.png)

比率分布暴露两簇异常：**ratio≈100**（金额被放大 100 倍，14 行）与 **ratio≈10⁻³**（数量被放大 1000 倍、金额实际正确，29 行）。正常行 ratio=1。


## 2. Repair —— 原则性修复

修复顺序遵循「**能精确还原就绝不统计猜测**」：

1. 标准化类别命名 → `Category_{id}`；
2. 去重：保留「最有效」的 `txn_id` 成员；
3. 数量离群 ÷1000；金额 ×100 错误重算为 `qty×price`；
4. 缺失 qty / price 由 `line_amount` 不变量**精确反推**；
5. 缺失日期由 `txn_id` 分块**精确还原**（时间感知）；
6. `store_id=9999` 由门店日循环序**精确还原**（结构感知）；
7. `product_id=9999` 无信号 → 种子化按经验频数采样（保持边际分布无偏）。


In [17]:
clean, change_log = hu.repair(dfs)
cl = pd.DataFrame(change_log)
print(cl[['step','table','column','operation','rows']].to_string(index=False))


 step      table      column                             operation  rows
    0 categories        name                     standardize label     2
    1      sales      txn_id         deduplicate (keep-most-valid)    50
    2      sales         qty                   scale outlier /1000    29
    3      sales line_amount              recompute qty*unit_price    13
    4      sales         qty      recover = line_amount/unit_price    56
    5      sales  unit_price             recover = line_amount/qty    56
    6      sales    txn_date      recover via txn_id block (ffill)    29
    7      sales    store_id         recover via store block order    13
    8      sales  product_id impute (empirical-frequency sampling)    28
    9      sales line_amount            normalize = qty*unit_price     0


## 3. Validate —— 完整性验证

对清洗结果跑 18 项引用完整性 / 越界 / 一致性检查。


In [19]:
checks = hu.validate(clean)
cdf = pd.DataFrame([(k, 'PASS' if v else 'FAIL') for k, v in checks.items()], columns=['check', 'status'])
print(cdf.to_string(index=False))
print('\nFAILED:', sum(1 for v in checks.values() if not v), '/', len(checks))


             check status
  no_null_txn_date   PASS
       no_null_qty   PASS
     no_null_price   PASS
    no_null_amount   PASS
     no_dup_txn_id   PASS
      txn_id_dense   PASS
  store_refs_valid   PASS
product_refs_valid   PASS
        qty_in_1_4   PASS
       qty_integer   PASS
    price_positive   PASS
 amount_consistent   PASS
    date_in_window   PASS
date_monotonic_tid   PASS
    products_clean   PASS
     vendors_clean   PASS
      stores_clean   PASS
  categories_clean   PASS

FAILED: 0 / 18


### 前后对比（每日营收）


In [21]:
def daily(sales, key='line_amount'):
    t = sales.copy()
    t['d'] = pd.to_datetime(t['txn_date'], errors='coerce').dt.normalize()
    t['x'] = pd.to_numeric(t[key], errors='coerce')
    return t.dropna(subset=['d']).groupby('d')['x'].sum()
raw_d = daily(dfs['sales'])
clean_d = daily(clean['sales'])
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(raw_d.index, raw_d.values, color='#c0392b', lw=1.1, alpha=0.75, label='raw')
ax.plot(clean_d.index, clean_d.values, color='#1b4f72', lw=1.6, label='clean')
ax.set_title('Daily total revenue — raw vs cleaned'); ax.legend(); ax.set_ylabel('revenue ($)')
plt.tight_layout(); plt.savefig(FIG / 'fig4_daily_revenue.png', dpi=110); plt.close()
print(f'raw  total revenue = {raw_d.sum():,.0f}')
print(f'clean total revenue = {clean_d.sum():,.0f}')
print(f'cleaning removes {raw_d.sum()-clean_d.sum():,.0f} of distortion ({(raw_d.sum()/clean_d.sum()-1)*100:.1f}% over)')


raw  total revenue = 263,530
clean total revenue = 183,483
cleaning removes 80,047 of distortion (43.6% over)


![每日营收对比](../reports/figures/fig4_daily_revenue.png)

原始营收被金额 ×100 与重复计费抬高了约 45%；清洗后恢复到真实水平，且**无任何一日的曲线形状被扭曲**。


## 4. Summary —— 摘要

- 检测到 **9 类**注入错误，全部被修复或明确说明；
- `qty`/`unit_price`/`txn_date`/`store_id` 四项是**精确还原**（0 猜测），`product_id` 是唯一无信号的列、以种子化分布保持采样填补；
- 18 项完整性检查 **全部通过**；
- 每日营收曲线清洗后被拉回真实水位，验证「清洗没有扭曲关键模式」。
